# Setting Up the Dataset

## How the whole team accesses the data

The MRI images are **not stored in GitHub**. They are too large for git, so `data/` is listed in `.gitignore`. That means:

| What you get from `git clone` / `git pull` | What you do **not** get |
|---|---|
| This notebook, training code, outlines | The actual image files |

**Every teammate downloads the same public dataset on their own machine** (or cloud runtime) by running this notebook. Everyone ends up with the same folder layout under `data/`, but each person creates that folder locally — git does not copy it for you.

### Workflow for a new contributor

1. Clone the repo and create your feature branch (see `README.md`).
2. Create a free [Kaggle](https://www.kaggle.com) account and an API token ([Account Settings](https://www.kaggle.com/settings/account)).
3. Run **all cells** in this notebook once. It downloads from Kaggle into `data/` next to the repo.
4. Open any other project notebook (`tumor-classification.ipynb`, etc.) and load images from `data/Training/` and `data/Testing/`.

Jeremy, Marco, Tyler, and Odon all use the **same source** ([Brain Tumor MRI Dataset on Kaggle](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/data)); no one needs to email zip files or commit images to the repo.

### Where files live after download

```
tumor-classification/
  data/                 ← created by this notebook (not in git)
    Training/
      glioma/
      meningioma/
      notumor/
      pituitary/
    Testing/
      (same four class folders)
```

### Using the data in other notebooks

After this notebook has run successfully, other notebooks should read from:

- `data/Training/` — model training and validation splits
- `data/Testing/` — held-out evaluation only

Copy the path setup from the last cells here, or use:

```python
from pathlib import Path
DATA_DIR = Path("data")  # relative to project root when cwd is the repo
TRAIN_DIR = DATA_DIR / "Training"
TEST_DIR = DATA_DIR / "Testing"
```

### Google Colab or a remote VM

Colab and similar environments start with an **empty disk**. Whoever uses that environment must run this notebook **in that session** (or copy `data/` from Google Drive if you saved it there). Closing Colab does not keep `data/` unless you persist it yourself.

---

## About this notebook

This notebook downloads the [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/data) from Kaggle into `data/`.

**Dataset:** `masoudnickparvar/brain-tumor-mri-dataset`
**Classes:** `glioma`, `meningioma`, `notumor`, `pituitary`
**Splits:** `Training/` and `Testing/` folders per class

Run it **once per machine or runtime** before any notebook that loads images.

## Prerequisites

You need a Kaggle account and API credentials:

1. Go to [Kaggle Account Settings](https://www.kaggle.com/settings/account) and create an API token.
2. Use **one** of the options below:
   - **Local:** save the token as `~/.kaggle/kaggle.json` and run `chmod 600 ~/.kaggle/kaggle.json`
   - **Remote / CI:** set `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables
   - **Google Colab:** upload `kaggle.json` in the credentials cell below

In [4]:
%pip install -q kaggle pandas


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import pandas as pd

DATASET_SLUG = "masoudnickparvar/brain-tumor-mri-dataset"
CLASSES = ("glioma", "meningioma", "notumor", "pituitary")
SPLITS = ("Training", "Testing")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / ".git").exists():
    # Useful when the notebook is opened from a subfolder in some remote environments.
    for parent in Path.cwd().resolve().parents:
        if (parent / ".git").exists():
            PROJECT_ROOT = parent
            break

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: /Users/jeremykelly/dev/class/ml/tumor-classification
Data directory: /Users/jeremykelly/dev/class/ml/tumor-classification/data


## Configure Kaggle credentials

Skip the Colab upload cell if you already have `~/.kaggle/kaggle.json` or environment variables set.

In [6]:
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"


def credentials_ready() -> bool:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    return KAGGLE_JSON.is_file()


def setup_colab_credentials() -> None:
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab; skipping upload prompt.")
        return

    if credentials_ready():
        print("Kaggle credentials already available.")
        return

    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("Expected to upload a file named kaggle.json")

    KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
    KAGGLE_JSON.write_bytes(uploaded["kaggle.json"])
    KAGGLE_JSON.chmod(0o600)
    print(f"Saved credentials to {KAGGLE_JSON}")


setup_colab_credentials()

if not credentials_ready():
    raise RuntimeError(
        "Kaggle credentials not found.\n"
        "Set KAGGLE_USERNAME and KAGGLE_KEY, or create ~/.kaggle/kaggle.json."
    )

print("Kaggle credentials detected.")

Not running in Colab; skipping upload prompt.


RuntimeError: Kaggle credentials not found.
Set KAGGLE_USERNAME and KAGGLE_KEY, or create ~/.kaggle/kaggle.json.

## Download and extract the dataset

Downloads from Kaggle into `data/`. Re-running this cell is safe: it skips download if the expected folders already exist unless `FORCE_DOWNLOAD = True`.

In [7]:
FORCE_DOWNLOAD = False


def dataset_layout_complete(root: Path) -> bool:
    return all((root / split / class_name).is_dir() for split in SPLITS for class_name in CLASSES)


def download_dataset(output_dir: Path, force: bool = False) -> None:
    if dataset_layout_complete(output_dir) and not force:
        print(f"Dataset already present in {output_dir}. Set FORCE_DOWNLOAD = True to re-download.")
        return

    if force and output_dir.exists():
        shutil.rmtree(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    if shutil.which("kaggle") is None:
        raise RuntimeError("Kaggle CLI not available after install. Restart the kernel and rerun.")

    print(f"Downloading {DATASET_SLUG} ...")
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            DATASET_SLUG,
            "-p",
            str(output_dir),
        ],
        check=True,
    )

    zip_files = sorted(output_dir.glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError(f"No zip archive found in {output_dir} after download.")

    archive = zip_files[0]
    print(f"Extracting {archive.name} ...")
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(output_dir)
    archive.unlink()
    print("Download and extraction complete.")


download_dataset(DATA_DIR, force=FORCE_DOWNLOAD)

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)


100%|██████████| 157M/157M [00:32<00:00, 5.00MB/s] 



Extracting brain-tumor-mri-dataset.zip ...
Download and extraction complete.


## Verify dataset structure

Confirms the expected folders exist and reports image counts per split and class.

In [8]:
def count_images(folder: Path) -> int:
    if not folder.is_dir():
        return 0
    return sum(1 for path in folder.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)


rows = []
for split in SPLITS:
    for class_name in CLASSES:
        class_dir = DATA_DIR / split / class_name
        rows.append(
            {
                "split": split,
                "class": class_name,
                "path": str(class_dir),
                "exists": class_dir.is_dir(),
                "image_count": count_images(class_dir),
            }
        )

summary = pd.DataFrame(rows)
display(summary)

if not summary["exists"].all():
    missing = summary.loc[~summary["exists"], "path"].tolist()
    raise FileNotFoundError(f"Missing expected folders:\n" + "\n".join(missing))

total_images = int(summary["image_count"].sum())
print(f"Total images found: {total_images:,}")
print("\nUse these paths in other notebooks:")
print(f"  DATA_DIR = Path('{DATA_DIR}')")
print(f"  TRAIN_DIR = DATA_DIR / 'Training'")
print(f"  TEST_DIR = DATA_DIR / 'Testing'")

,split,class,path,exists,image_count
0,Training,glioma,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,1400
1,Training,meningioma,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,1400
2,Training,notumor,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,1400
3,Training,pituitary,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,1400
4,Testing,glioma,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,400
5,Testing,meningioma,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,400
6,Testing,notumor,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,400
7,Testing,pituitary,/Users/jeremykelly/dev/class/ml/tumor-classifi...,True,400


Total images found: 7,200

Use these paths in other notebooks:
  DATA_DIR = Path('/Users/jeremykelly/dev/class/ml/tumor-classification/data')
  TRAIN_DIR = DATA_DIR / 'Training'
  TEST_DIR = DATA_DIR / 'Testing'


## Shared path constants

Other notebooks can copy these paths or import them after running this notebook in the same session.

In [9]:
TRAIN_DIR = DATA_DIR / "Training"
TEST_DIR = DATA_DIR / "Testing"

PATHS = {
    "project_root": PROJECT_ROOT,
    "data_dir": DATA_DIR,
    "train_dir": TRAIN_DIR,
    "test_dir": TEST_DIR,
    "classes": CLASSES,
    "splits": SPLITS,
}

PATHS

{'project_root': PosixPath('/Users/jeremykelly/dev/class/ml/tumor-classification'),
 'data_dir': PosixPath('/Users/jeremykelly/dev/class/ml/tumor-classification/data'),
 'train_dir': PosixPath('/Users/jeremykelly/dev/class/ml/tumor-classification/data/Training'),
 'test_dir': PosixPath('/Users/jeremykelly/dev/class/ml/tumor-classification/data/Testing'),
 'classes': ('glioma', 'meningioma', 'notumor', 'pituitary'),
 'splits': ('Training', 'Testing')}